# Librerías

In [1]:
import pandas as pd
import re

# Documentos

In [2]:
EAL_25 = '../../Data/processed/EAL/2024/EAL-25.csv'
EAL_24 = '../../Data/processed/EAL/2024/EAL-24.csv'
EAL_23 = '../../Data/processed/EAL/2024/EAL-23.csv'

EAL_22 = '../../Data/processed/EAL/2024/EAL-22.csv'
EAL_21 = '../../Data/processed/EAL/2024/EAL-21.csv'

# Funciones extracción

EAL-23 A 25

In [3]:
def read_csv_categorias(doc_path):
    df = (
        pd.read_csv(doc_path, encoding="utf-8")
        .iloc[1:]
        .reset_index(drop=True)
        .rename(columns={
            "ENCUESTA ANUAL LABORAL": "ambito",
            "Unnamed: 1": "TOTAL",
            "Unnamed: 2": "NADA",
            "Unnamed: 3": "POCO",
            "Unnamed: 4": "BASTANTE",
            "EAL": "MUCHO"
        })
    )

    return df

In [4]:
def transformation(df, anio=2024, filas_debajo_eliminar=2):
    df = df.copy()

    col_texto = "ambito"
    patron_tabla = r"^(EAL-\d+[A-Za-z]?)"

    # Detectar código de tabla: EAL-25, EAL-25a, EAL-25b...
    tabla_detectada = (
        df[col_texto]
        .astype(str)
        .str.extract(patron_tabla, expand=False)
    )

    # Filas donde empieza una tabla
    mask_tabla = tabla_detectada.notna()

    # Pregunta completa asociada a cada bloque
    pregunta_detectada = (
        df[col_texto]
        .where(mask_tabla)
        .ffill()
        .astype(str)
        .str.replace(r"^EAL-\d+[A-Za-z]?\.\s*", "", regex=True)
    )

    # Insertar columnas al principio
    df.insert(0, "anio", anio)
    df.insert(1, "tabla", tabla_detectada.ffill())
    df.insert(2, "pregunta", pregunta_detectada)

    # Eliminar fila de pregunta + N filas inferiores
    mask_eliminar = pd.concat(
        [
            mask_tabla.shift(i, fill_value=False)
            for i in range(filas_debajo_eliminar + 1)
        ],
        axis=1
    ).any(axis=1)

    df = df.loc[~mask_eliminar].reset_index(drop=True)

    return df

In [5]:
def process_categorias(doc_path, anio=2024, filas_debajo_eliminar=2):
    df = read_csv_categorias(doc_path)
    df = transformation(
        df,
        anio=anio,
        filas_debajo_eliminar=filas_debajo_eliminar
    )

    return df

Función para formato largo

In [6]:
def to_long_format(
    df,
    id_vars,
    value_vars,
    var_name,
    value_name="porcentaje"
):
    df = df.copy()

    df_long = df.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name=var_name,
        value_name=value_name
    )

    df_long[value_name] = (
        df_long[value_name]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace("%", "", regex=False)
    )

    df_long[value_name] = pd.to_numeric(
        df_long[value_name],
        errors="coerce"
    )

    df_long = df_long.dropna(subset=[value_name]).reset_index(drop=True)

    return df_long

Read csv con una sola tabla (EAL-21, EAL-22)

In [7]:
def read_csv_eal(doc_path):
    df = pd.read_csv(doc_path, encoding="utf-8", header=None)

    df = df.map(
        lambda x: re.sub(r"\s+", " ", str(x)).strip()
        if pd.notna(x) else pd.NA
    )

    df = df.replace(["", "nan", "NaN"], pd.NA)

    return df

EAL-22 CCAA

In [8]:
def process_ccaa_long(df, anio=2024):
    df = df.copy()

    # -----------------------------
    # 1. Extract table and question
    # -----------------------------
    values = df.stack().dropna().astype(str)

    titulo = values[
        values.str.match(r"^EAL-\d+[A-Za-z]?\.", na=False)
        ].iloc[0]

    tabla = re.match(r"^(EAL-\d+[A-Za-z]?)", titulo).group(1)

    pregunta = re.sub(
        r"^EAL-\d+[A-Za-z]?\.\s*",
        "",
        titulo
    )

    # -----------------------------
    # 2. Detect CCAA header rows
    # -----------------------------
    ccaa_keywords = [
        "TOTAL", "ANDALUCÍA", "ARAGÓN", "ASTURIAS (PRINCIPADO DE)", "BALEARES (ILLES)",
        "CANARIAS", "CANTABRIA", "CASTILLA-LA MANCHA", "CASTILLA Y LEÓN", "CATALUÑA",
        "COMUNITAT VALENCIANA", "EXTREMADURA", "GALICIA", "MADRID (COMUNIDAD DE)",
        "MURCIA (REGIÓN DE)", "NAVARRA (COMUNIDAD FORAL DE)", "PAÍS VASCO", "RIOJA, LA"
    ]

    patron_ccaa = "|".join(re.escape(x) for x in ccaa_keywords)

    mask_headers = df.apply(
        lambda row: row.astype(str)
        .str.contains(patron_ccaa, case=False, regex=True)
        .sum() >= 2,
        axis=1
    )

    header_rows = df.index[mask_headers].tolist()

    # -----------------------------
    # 3. Process each horizontal block
    # -----------------------------
    bloques = []
    col_competencia = df.columns[0]

    for i, header_row in enumerate(header_rows):

        start_row = header_row + 1

        if i + 1 < len(header_rows):
            end_row = header_rows[i + 1]
        else:
            end_row = len(df)

        header = df.loc[header_row]

        value_cols = header[header.notna()].index.tolist()
        value_cols = [c for c in value_cols if c != col_competencia]

        nombres_ccaa = header[value_cols].tolist()

        block = df.loc[start_row:end_row - 1, [col_competencia] + value_cols].copy()

        block.columns = ["competencia"] + nombres_ccaa

        block = block[block["competencia"].notna()].copy()

        block_long = block.melt(
            id_vars="competencia",
            var_name="comunidad_autonoma",
            value_name="porcentaje"
        )

        bloques.append(block_long)

    # -----------------------------
    # 4. Concatenate and clean values
    # -----------------------------
    df_final = pd.concat(bloques, ignore_index=True)

    df_final["porcentaje"] = (
        df_final["porcentaje"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace("%", "", regex=False)
    )

    df_final["porcentaje"] = pd.to_numeric(
        df_final["porcentaje"],
        errors="coerce"
    )

    df_final = df_final.dropna(subset=["porcentaje"]).reset_index(drop=True)

    # -----------------------------
    # 5. Add metadata
    # -----------------------------
    df_final.insert(0, "anio", anio)
    df_final.insert(1, "tabla", tabla)
    df_final.insert(2, "pregunta", pregunta)

    return df_final

EAL-21 SECTORES

In [9]:
def process_sector_wide(df, anio=2024):
    df = df.copy()

    # 1. Extraer tabla y pregunta
    values = df.stack().dropna().astype(str)

    titulo = values[
        values.str.match(r"^EAL-\d+[A-Za-z]?\.", na=False)
    ].iloc[0]

    tabla = re.match(r"^(EAL-\d+[A-Za-z]?)", titulo).group(1)

    pregunta = re.sub(
        r"^EAL-\d+[A-Za-z]?\.\s*",
        "",
        titulo
    )

    # 2. Detectar fila de cabecera
    columnas_sector = ["TOTAL", "INDUSTRIA", "CONSTRUCCIÓN", "SERVICIOS"]
    patron_sector = "|".join(re.escape(x) for x in columnas_sector)

    mask_header = df.apply(
        lambda row: row.astype(str)
        .str.contains(patron_sector, case=False, regex=True)
        .sum() >= 2,
        axis=1
    )

    header_row = df.index[mask_header][0]

    # 3. Detectar columnas donde están TOTAL, INDUSTRIA, etc.
    header = df.loc[header_row]

    value_cols = header[
        header.astype(str).str.contains(
            patron_sector,
            case=False,
            regex=True,
            na=False
        )
    ].index.tolist()

    nombres_columnas = header[value_cols].tolist()

    # 4. Extraer bloque de datos
    col_competencia = df.columns[0]

    df_sector = df.loc[
        header_row + 1:,
        [col_competencia] + value_cols
    ].copy()

    df_sector.columns = ["ambito"] + nombres_columnas

    # 5. Eliminar filas vacías o notas al pie
    df_sector = df_sector[df_sector["ambito"].notna()].copy()

    # 6. Convertir columnas numéricas
    for col in nombres_columnas:
        df_sector[col] = (
            df_sector[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .str.replace("%", "", regex=False)
        )

        df_sector[col] = pd.to_numeric(df_sector[col], errors="coerce")

    # Eliminar filas que no tengan datos numéricos en TOTAL
    df_sector = df_sector.dropna(subset=["TOTAL"]).reset_index(drop=True)

    # 7. Insertar metadatos
    df_sector.insert(0, "anio", anio)
    df_sector.insert(1, "tabla", tabla)
    df_sector.insert(2, "pregunta", pregunta)

    return df_sector

# Procesamiento

## EAL-21

In [10]:
df_21_raw = read_csv_eal(EAL_21)
df_21 = process_sector_wide(df_21_raw, anio=2024)

df_21.head()

,anio,tabla,pregunta,ambito,TOTAL,INDUSTRIA,CONSTRUCCIÓN,SERVICIOS
0,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De dirección,16.347340,16.486105,15.703615,16.439153
1,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De trabajo en equipo,28.494848,24.620076,24.630091,30.078036
2,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De atención al público/ trato a clientes,24.530267,9.157845,5.801208,31.456162
3,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Administrativas de oficina,17.027644,19.208171,14.684754,16.994533
4,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Técnicas específicas del puesto de trabajo,50.838810,64.154498,66.288887,44.985817


In [11]:
df_21_long = to_long_format(
    df=df_21,
    id_vars=["anio", "tabla", "pregunta", "ambito"],
    value_vars=["TOTAL", "INDUSTRIA", "CONSTRUCCIÓN", "SERVICIOS"],
    var_name="sector",
    value_name="porcentaje"
)

In [12]:
df_21_long

,anio,tabla,pregunta,ambito,sector,porcentaje
0,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De dirección,TOTAL,16.347340
1,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De trabajo en equipo,TOTAL,28.494848
2,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De atención al público/ trato a clientes,TOTAL,24.530267
3,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Administrativas de oficina,TOTAL,17.027644
4,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Técnicas específicas del puesto de trabajo,TOTAL,50.838810
5,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De resolución de problemas (localización de pr...,TOTAL,11.209969
6,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,En lenguas extranjeras,TOTAL,7.478712
7,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Generales de tecnologías de la información,TOTAL,12.666991
8,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Profesionales de tecnologías de la información,TOTAL,5.310919
9,2024,EAL-21,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Básicas de cálculo y/o comunicación oral o esc...,TOTAL,1.254936


**Validación**
- 11 preguntas con 4 sectores equivalen a las 44 filas del df

## EAL-22

In [13]:
df_22_raw = read_csv_eal(EAL_22)
df_22_long = process_ccaa_long(df_22_raw, anio=2024)

In [14]:
df_22_long

,anio,tabla,pregunta,competencia,comunidad_autonoma,porcentaje
0,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De dirección,TOTAL,16.347340
1,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De trabajo en equipo,TOTAL,28.494848
2,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,De atención al público/ trato a clientes,TOTAL,24.530267
3,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Administrativas de oficina,TOTAL,17.027644
4,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Técnicas específicas del puesto de trabajo,TOTAL,50.838810
...,...,...,...,...,...,...
193,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,En lenguas extranjeras,"RIOJA, LA",5.816555
194,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Generales de tecnologías de la información,"RIOJA, LA",13.255034
195,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Profesionales de tecnologías de la información,"RIOJA, LA",3.131991
196,2024,EAL-22,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Básicas de cálculo y/o comunicación oral o esc...,"RIOJA, LA",1.957494


In [15]:
df_22_long['comunidad_autonoma'].unique() 

<StringArray>
[                       'TOTAL',                    'ANDALUCÍA',
                       'ARAGÓN',     'ASTURIAS (PRINCIPADO DE)',
              'BALEARS (ILLES)',                     'CANARIAS',
                    'CANTABRIA',           'CASTILLA-LA MANCHA',
              'CASTILLA Y LEÓN',                     'CATALUÑA',
         'COMUNITAT VALENCIANA',                  'EXTREMADURA',
                      'GALICIA',        'MADRID (COMUNIDAD DE)',
           'MURCIA (REGIÓN DE)', 'NAVARRA (COMUNIDAD FORAL DE)',
                   'PAÍS VASCO',                    'RIOJA, LA']
Length: 18, dtype: str

**Validación**
- 18 CCAA en el documento excel
- 11 preguntas con 18 CCAA equivalen a las 198 filas del df

## EAL-23

In [16]:
df_23 = process_categorias(EAL_23)

In [17]:
df_23

,anio,tabla,pregunta,ambito,TOTAL,NADA,POCO,BASTANTE,MUCHO
0,2024,EAL-23,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Responder a un sistema predeterminado de promo...,100,24.833,42.321,26.86,5.987
1,2024,EAL-23,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Readaptar al personal a los cambios técnicos i...,100,12.615,22.193,49.421,15.771
2,2024,EAL-23,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Readaptar al personal a los cambios organizati...,100,14.246,29.411,44.323,12.02
3,2024,EAL-23,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Adaptar al personal recién incorporado a las t...,100,7.888,15.038,50.533,26.541
4,2024,EAL-23,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Mejorar la cualificación básica del personal p...,100,8.168,22.567,51.043,18.222
5,2024,EAL-23,EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TR...,Adaptación impuesta por cambios normativos,100,8.955,22.904,42.755,25.385
6,2024,EAL-23a,EMPRESAS DE 5 A 49 TRABAJADORES QUE PROPORCION...,Responder a un sistema predeterminado de promo...,100,26.215,42.352,25.663,5.77
7,2024,EAL-23a,EMPRESAS DE 5 A 49 TRABAJADORES QUE PROPORCION...,Readaptar al personal a los cambios técnicos i...,100,13.406,22.523,48.979,15.093
8,2024,EAL-23a,EMPRESAS DE 5 A 49 TRABAJADORES QUE PROPORCION...,Readaptar al personal a los cambios organizati...,100,14.992,29.748,43.875,11.385
9,2024,EAL-23a,EMPRESAS DE 5 A 49 TRABAJADORES QUE PROPORCION...,Adaptar al personal recién incorporado a las t...,100,8.366,15.287,50.501,25.846


In [18]:
df_23['tabla'].unique()

<StringArray>
['EAL-23', 'EAL-23a', 'EAL-23b', 'EAL-23c', 'EAL-23d', 'EAL-23e', 'EAL-23f']
Length: 7, dtype: str

**Validación**  
- Comprobado el número de filas finales con el excel inicial (7 tablas de 6 filas de resultados)
- Los decimales son correctos, en el excel solo se muestra un decimal pero si se mueve la coma se ven todos los que aparecen en el df importado

## EAL-24

In [19]:
df_24 = process_categorias(EAL_24)

In [20]:
df_24['tabla'].unique()

<StringArray>
['EAL-24', 'EAL-24a', 'EAL-24b', 'EAL-24c', 'EAL-24d', 'EAL-24e', 'EAL-24f']
Length: 7, dtype: str

**Validación**  
- Comprobado el número de filas finales con el excel inicial (6 tablas de 10 filas de resultados)
- Los decimales son correctos, en el excel solo se muestra un decimal pero si se mueve la coma se ven todos los que aparecen en el df importado

## EAL-25

In [21]:
df_25 = process_categorias(EAL_25)

In [22]:
df_25

,anio,tabla,pregunta,ambito,TOTAL,NADA,POCO,BASTANTE,MUCHO
0,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,El nivel de formación de los empleados era el ...,100,7.497,11.745,48.18,32.578
1,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,La empresa prefirió contratar a personal con l...,100,16.991,14.927,45.484,22.598
2,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,Dificultad para evaluar la necesidades de form...,100,33.929,46.144,15.427,4.501
3,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,La falta de cursos de formación adecuados en e...,100,32.974,40.854,18.727,7.444
4,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,El alto coste de los cursos de formación,100,32.154,34.644,22.305,10.896
5,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,La empresa prefirió tener personal con contrat...,100,69.386,20.775,7.963,1.875
6,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,El mayor esfuerzo realizado en años anteriores...,100,42.478,33.815,18.795,4.912
7,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,La elevada carga de trabajo y el escaso tiempo...,100,23.112,29.498,30.973,16.417
8,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,Dificultad para acceder a ayudas o subvencione...,100,36.386,31.367,20.381,11.866
9,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,Otras razones,100,63.988,21.749,7.994,6.269


In [23]:
df_25['TOTAL'].unique()

<StringArray>
['100']
Length: 1, dtype: str

In [24]:
df_25['tabla'].unique()

<StringArray>
['EAL-25', 'EAL-25a', 'EAL-25b', 'EAL-25c', 'EAL-25d', 'EAL-25e']
Length: 6, dtype: str

In [25]:
df_25_long = to_long_format(
    df=df_25,
    id_vars=["anio", "tabla", "pregunta", "ambito"],
    value_vars=["NADA", "POCO", "BASTANTE", "MUCHO"],
    var_name="categoria",
    value_name="porcentaje"
)

In [26]:
df_25_long

,anio,tabla,pregunta,ambito,categoria,porcentaje
0,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,El nivel de formación de los empleados era el ...,NADA,7.497
1,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,La empresa prefirió contratar a personal con l...,NADA,16.991
2,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,Dificultad para evaluar la necesidades de form...,NADA,33.929
3,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,La falta de cursos de formación adecuados en e...,NADA,32.974
4,2024,EAL-25,EMPRESAS QUE NO PROPORCIONARON FORMACIÓN A SUS...,El alto coste de los cursos de formación,NADA,32.154
...,...,...,...,...,...,...
235,2024,EAL-25e,EMPRESAS DEL SECTOR SERVICIOS QUE NO PROPORCIO...,La empresa prefirió tener personal con contrat...,MUCHO,1.792
236,2024,EAL-25e,EMPRESAS DEL SECTOR SERVICIOS QUE NO PROPORCIO...,El mayor esfuerzo realizado en años anteriores...,MUCHO,4.502
237,2024,EAL-25e,EMPRESAS DEL SECTOR SERVICIOS QUE NO PROPORCIO...,La elevada carga de trabajo y el escaso tiempo...,MUCHO,17.522
238,2024,EAL-25e,EMPRESAS DEL SECTOR SERVICIOS QUE NO PROPORCIO...,Dificultad para acceder a ayudas o subvencione...,MUCHO,12.699


In [27]:
df_25_long['categoria'].unique()

<StringArray>
['NADA', 'POCO', 'BASTANTE', 'MUCHO']
Length: 4, dtype: str

La categoria total siempre es 100, así que se elimina

**Validación**  
- Comprobado el número de filas finales con el excel inicial (6 tablas de 10 filas de resultados)
- Los decimales son correctos, en el excel solo se muestra un decimal pero si se mueve la coma se ven todos los que aparecen en el df importado